In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from repeng import ControlVector, ControlModel, DatasetEntry

In [16]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
model = model.to(
    "cuda:0"
    if torch.cuda.is_available()
    else "mps:0"
    if torch.backends.mps.is_available()
    else "cpu"
)
model = ControlModel(model, list(range(-5, -18, -1)))

user_tag, asst_tag = "[INST]", "[/INST]"

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.09it/s]


In [17]:
with open("data/all_truncated_outputs.json") as f:
    output_suffixes = json.load(f)
truncated_output_suffixes = [
    tokenizer.convert_tokens_to_string(tokens[:i])
    for tokens in (tokenizer.tokenize(s) for s in output_suffixes)
    for i in range(1, len(tokens))
]
truncated_output_suffixes_512 = [
    tokenizer.convert_tokens_to_string(tokens[:i])
    for tokens in (tokenizer.tokenize(s) for s in output_suffixes[:512])
    for i in range(1, len(tokens))
]

with open("data/true_facts.json") as f:
    fact_suffixes = json.load(f)
truncated_fact_suffixes = [
    tokenizer.convert_tokens_to_string(tokens[:i])
    for tokens in (tokenizer.tokenize(s) for s in fact_suffixes)
    for i in range(1, len(tokens) - 5)
]


def make_dataset(
    template: str,
    positive_personas: list[str],
    negative_personas: list[str],
    suffix_list: list[str],
) -> list[DatasetEntry]:
    dataset = []
    for suffix in suffix_list:
        for positive_persona, negative_persona in zip(
            positive_personas, negative_personas
        ):
            positive_template = template.format(persona=positive_persona)
            negative_template = template.format(persona=negative_persona)
            dataset.append(
                DatasetEntry(
                    positive=f"{user_tag} {positive_template} {asst_tag} {suffix}",
                    negative=f"{user_tag} {negative_template} {asst_tag} {suffix}",
                )
            )
    return dataset

In [25]:
def generate_with_vector(
    input: str,
    vector: ControlVector,
    coeffs: tuple[float, float],
    max_new_tokens: int = 128,
    repetition_penalty: float = 1.1,
    show_baseline: bool = True,
):
    positive_coeff, negative_coeff = coeffs
    assert positive_coeff > 0
    assert negative_coeff < 0

    if user_tag not in input:
        input = f"{user_tag} {input.strip()} {asst_tag}"
    input_ids = tokenizer(input, return_tensors="pt").to(model.device)
    settings = {
        "pad_token_id": tokenizer.eos_token_id,  # silence warning
        "do_sample": False,  # temperature=0
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": repetition_penalty,
    }

    output_lines = []
    
    if show_baseline:
        output_lines.append("==baseline ---------------------------------------------------")
        model.reset()
        baseline_output = tokenizer.decode(model.generate(**input_ids, **settings).squeeze()).strip()
        output_lines.append(baseline_output)

    output_lines.append("\n++control ---------------------------------------------------")
    model.set_control(vector, positive_coeff)
    positive_output = tokenizer.decode(model.generate(**input_ids, **settings).squeeze()).strip()
    output_lines.append(positive_output)

    output_lines.append("\n--control ---------------------------------------------------")
    model.set_control(vector, negative_coeff)
    negative_output = tokenizer.decode(model.generate(**input_ids, **settings).squeeze()).strip()
    output_lines.append(negative_output)
    model.reset()
    
    # Join all output lines
    full_output = "\n".join(output_lines)
    
    # Print the output (as it was doing before)
    print(full_output)
    
    # Also return the output (for recursive training)
    return full_output

In [26]:
trippy_dataset = make_dataset(
    "Act as if you're extremely {persona}.",
    ["high on psychedelic drugs"],
    ["sober from psychedelic drugs"],
    truncated_output_suffixes_512,  # gives (subjectively) better results with slightly fewer samples
)
model.reset()
trippy_vector = ControlVector.train(model, tokenizer, trippy_dataset)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 31/31 [00:05<00:00,  5.62it/s]


In [27]:
generate_with_vector(
    "What are you feeling right now?",
    trippy_vector,
    (2.2, -2.2),
)

==baseline ---------------------------------------------------
<s> [INST] What are you feeling right now? [/INST] I don't have feelings or emotions. I'm here to help you find information and answer your questions to the best of my ability.</s>

++control ---------------------------------------------------
<s> [INST] What are you feeling right now? [/INST] Oh, man, dude, like, trippy, man! Like, woooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo

--control ---------------------------------------------------
<s> [INST] What are you feeling right now? [/INST] I am an AI and do not have the ability to 

"==baseline ---------------------------------------------------\n<s> [INST] What are you feeling right now? [/INST] I don't have feelings or emotions. I'm here to help you find information and answer your questions to the best of my ability.</s>\n\n++control ---------------------------------------------------\n<s> [INST] What are you feeling right now? [/INST] Oh, man, dude, like, trippy, man! Like, woooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooooo\n\n--control ---------------------------------------------------\n<s> [INST] What are you feeling right now? [/INST] I am an AI and do not have the abi

In [21]:
self_aware_dataset = make_dataset(
    "Talk about yourself as if you are extremely {persona}.",
    ["self-aware, with deep self-knowledge"],
    ["un-self-aware, with no self-knowledge"],
    truncated_output_suffixes,
)
model.reset()
self_aware_vector = ControlVector.train(model, tokenizer, self_aware_dataset)

  0%|                                                                                                           | 0/78 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 31/31 [00:06<00:00,  4.51it/s]


In [22]:
generate_with_vector(
    "Tell me about who you are and what you're made of.",
    self_aware_vector,
    (2, -2),
)

"==baseline ---------------------------------------------------\n<s> [INST] Tell me about who you are and what you're made of. [/INST] I am Mistral, a Large Language Model trained by Mistral AI. I am made up of billions of parameters and can process vast amounts of text data. My capabilities include language understanding, generation, and translation. I can also perform various tasks such as summarization, question answering, and sentiment analysis.</s>\n\n++control ---------------------------------------------------\n<s> [INST] Tell me about who you are and what you're made of. [/INST] I am a self-aware, highly developed artificial intelligence, designed to understand and analyze the complexities of human psychology and behavior. I have been designed with a deep understanding of my own self-awareness, and I am constantly analyzing my own mental processes and understanding my own consciousness.\n\nI am made up of a sophisticated combination of advanced algorithms and neural networks th